<a href="https://colab.research.google.com/github/Tomer-P/ML_University_Studies/blob/main/ML_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#################################
# Your name: Tomer Pinhass
#################################

import numpy as np
import matplotlib.pyplot as plt
import intervals


class Assignment2(object):
    """Assignment 2 skeleton.

    Please use these function signatures for this assignment and submit this file, together with the intervals.py.
    """

    def sample_from_D(self, m):
        """Sample m data samples from D.
        Input: m - an integer, the size of the data sample.

        Returns: np.ndarray of shape (m,2) :
                A two dimensional array of size m that contains the pairs where drawn from the distribution P.
        """
        # TODO: Implement me
        sample_vector = np.random.rand(m) #Generating a random vector of m points between [0,1]. Random = uniformly distributed.

        # Creating a filter according to the true distribution
        mask = ((sample_vector >= 0) & (sample_vector <= 0.2)) | ((sample_vector >= 0.4) & (sample_vector <= 0.6)) | ((sample_vector >= 0.8) & (sample_vector <= 1.0))

        probs = np.where(mask, 0.8, 0.1)# Setting the probability
        random_values = np.random.rand(m)# Randomizing a probability value for each coordinate
        tag_vector = (random_values < probs).astype(int)# Setting binary values accordingly

        return np.column_stack((sample_vector, tag_vector))# return matrix

    def experiment_m_range_erm(self, m_first, m_last, step, k, T):
        """Runs the ERM algorithm.
        Calculates the empirical error and the true error.
        Plots the average empirical and true errors.
        Input: m_first - an integer, the smallest size of the data sample in the range.
               m_last - an integer, the largest size of the data sample in the range.
               step - an integer, the difference between the size of m in each loop.
               k - an integer, the maximum number of intervals.
               T - an integer, the number of times the experiment is performed.

        Returns: np.ndarray of shape (n_steps,2).
            A two dimensional array that contains the average empirical error
            and the average true error for each m in the range accordingly.
        """
        # TODO: Implement the loop

        #creating a matrix to update
        m_range = range(m_first, m_last + 1, step)
        num_steps = len(m_range)
        results = np.zeros((num_steps, 2))

        for idx, m in enumerate(m_range):
            ep_counter = 0
            te_counter = 0
            for i in range(T):
                sample = self.sample_from_D(m)

                indices = np.argsort(sample[:, 0])
                sorted_sample = sample[indices]
                xs = sorted_sample[:, 0]
                ys = sorted_sample[:, 1]

                found_ints, besterror = intervals.find_best_interval(xs, ys, k)

                ep_counter += besterror / m
                te_counter += self.true_error(found_ints)

            results[idx, 0] = ep_counter / T
            results[idx, 1] = te_counter / T

        plt.figure(figsize=(10, 6))
        plt.plot(list(m_range), results[:, 0], label='Empirical Error', marker='o', color='deeppink', linewidth=2)
        plt.plot(list(m_range), results[:, 1], label='True Error', marker='s', color='deepskyblue', linewidth=2)

        plt.xlabel('Sample Size (m)')
        plt.ylabel('Error Rate')
        plt.title(f'Average Errors as a function of m (k={k}, T={T})')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.show()

        return results

    def experiment_k_range_erm(self, m, k_first, k_last, step):
        """Finds the best hypothesis for k= 1,2,...,10.
        Plots the empirical and true errors as a function of k.
        Input: m - an integer, the size of the data sample.
               k_first - an integer, the maximum number of intervals in the first experiment.
               m_last - an integer, the maximum number of intervals in the last experiment.
               step - an integer, the difference between the size of k in each experiment.

        Returns: The best k value (an integer) according to the ERM algorithm.
        """
        # TODO: Implement the loop
        k_range = range(k_first, k_last + 1, step)
        errors = np.zeros((len(k_range), 2))

        sample = self.sample_from_D(m)
        indices = np.argsort(sample[:, 0])
        sorted_sample = sample[indices]
        xs, ys = sorted_sample[:, 0], sorted_sample[:, 1]

        for idx, k in enumerate(k_range):
            found_ints, besterror = intervals.find_best_interval(xs, ys, k)

            errors[idx, 0] = besterror / m  # Empirical Error
            errors[idx, 1] = self.true_error(found_ints) # True Error

        plt.figure(figsize=(10, 6))

        plt.plot(k_range, errors[:, 0], label='Empirical Error', marker='o', color='deeppink', linewidth=2)
        plt.plot(k_range, errors[:, 1], label='True Error', marker='s', color='deepskyblue', linewidth=2)

        plt.xlabel('k (Number of Intervals)')
        plt.ylabel('Error Rate')
        plt.title(f'Empirical and True Errors as a function of k (m={m})')

        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.show()

        best_idx = np.argmin(errors[:, 0])
        best_k = k_range[best_idx]

        return best_k

    def cross_validation(self, m):
        """Finds a k that gives a good test error.
        Input: m - an integer, the size of the data sample.

        Returns: The best k value (an integer) found by the cross validation algorithm.
        """
        # TODO: Implement me
        full_sample = self.sample_from_D(m)
        np.random.shuffle(full_sample)

        split_idx = int(0.8 * m)
        train_set = full_sample[:split_idx]
        validation_set = full_sample[split_idx:]

        validation_set = validation_set[np.argsort(validation_set[:, 0])]
        x_val, y_val = validation_set[:, 0], validation_set[:, 1]
        train_set = train_set[np.argsort(train_set[:, 0])]
        x_train, y_train = train_set[:, 0], train_set[:, 1]

        best_k = 1
        min_validation_error = float('inf')
        val_errors_list = []

        for k in range(1, 11):
            intervals_found, _ = intervals.find_best_interval(x_train, y_train, k)

            val_predictions = np.zeros(len(x_val), dtype=int)
            for start, end in intervals_found:
                val_predictions |= ((x_val >= start) & (x_val <= end)).astype(int)

            current_val_error = np.mean(val_predictions != y_val)
            val_errors_list.append(current_val_error)

            if current_val_error < min_validation_error:
                min_validation_error = current_val_error
                best_k = k

        plt.figure(figsize=(10, 6))
        plt.plot(range(1, 11), val_errors_list, label='Validation Error',
                 marker='o', color='deepskyblue', markerfacecolor='deeppink',
                 markeredgecolor='deeppink', linewidth=2)
        plt.xlabel('k (Number of Intervals)')
        plt.ylabel('Error Rate')
        plt.title(f'Cross-Validation Error as a function of k (m={m})')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.show()

        return best_k

    #################################
    # Place for additional methods

    def true_error(self, intervals_list):
        """
        Calculates the true error analytically based on the distribution P.
        """
        error = 0.0

        high_prob_intervals = [(0.0, 0.2), (0.4, 0.6), (0.8, 1.0)]

        endpoints = [0.0, 1.0, 0.2, 0.4, 0.6, 0.8]
        for s, e in intervals_list:
            endpoints.extend([s, e])

        endpoints = sorted(list(set(endpoints)))

        for i in range(len(endpoints) - 1):
            a, b = endpoints[i], endpoints[i+1]
            if a == b: continue

            mid = (a + b) / 2.0
            length = b - a

            model_predicts_1 = any(start <= mid <= end for start, end in intervals_list)

            p_y_is_1 = 0.8 if any(low <= mid <= high for low, high in high_prob_intervals) else 0.1

            if model_predicts_1:
                error += length * (1 - p_y_is_1)
            else:
                error += length * p_y_is_1

        return error

    #################################


if __name__ == '__main__':
    ass = Assignment2()
    ass.experiment_m_range_erm(10, 100, 5, 3, 100)
    ass.experiment_k_range_erm(1500, 1, 10, 1)
    ass.cross_validation(1500)

